# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneha27patel/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# 1. Unit of Analysis: 1 Row = 1 Pseudonymized Content Item (URL/Page)
# 2. Table Used: content_refresh_anonymized.csv (Starter Dataset / Warehouse Dimension Slice)
# 3. Time Window: 90-day historical search performance window
# 4. Target / Proxy: target_priority_flag ((trend_direction == 'down') & (avg_position <= 20))
# 5. Deliberately Excluded: trend_pct (to prevent feature leakage)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# Join Key: content_id (pseudonymized content item identifier)
# Features (5 max): impressions_90d, avg_position, ctr, content_age_days, days_since_last_update
# Context: content_type, word_count
# Label / Target: target_priority_flag
# Excluded (Leakage Trap): trend_pct (because trend_direction is derived directly from trend_pct)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# Load dataset directly from URL
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Query 1: Verify Grain (1 Row = 1 Content ID)
print("--- Query 1: Grain Verification ---")
print(f"Total Rows: {len(df)} | Unique Content IDs: {df['content_id'].nunique()}")

# Query 2: Filter Available Rows
print("\n--- Query 2: Row Availability ---")
valid_rows = df[(df['impressions_90d'] > 0) & (df['avg_position'].notna())]
print(f"Rows with active search signals: {len(valid_rows)} / {len(df)}")

# Query 3: Target Distribution
df['target_priority_flag'] = (df['trend_direction'] == 'down') & (df['avg_position'] <= 20)
print("\n--- Query 3: Target Distribution ---")
print(df['target_priority_flag'].value_counts())

# Feature Frame (5 Features - all knowable at decision time)
features = ['impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update']
X = df[features].fillna(0)
y = df['target_priority_flag']

# Honest Model (Without Leakage)
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X, y)
preds = clf.predict(X)
print(f"\n--- Honest Model Precision: {precision_score(y, preds):.4f} ---")

# --- THE LEAKAGE TRAP EXPERIMENT ---
# Add label-derived feature 'trend_pct' on purpose to demonstrate leakage
X_leaked = X.copy()
X_leaked['leaked_trend_pct'] = df['trend_pct'].fillna(0)

clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaked.fit(X_leaked, y)
preds_leaked = clf_leaked.predict(X_leaked)
print(f"--- TRAP Model Precision (With Leaked Feature): {precision_score(y, preds_leaked):.4f} ---")
print("Trap sprung! The score jumped to near-perfect because 'trend_pct' leaks the target label.")

--- Query 1: Grain Verification ---
Total Rows: 30000 | Unique Content IDs: 30000

--- Query 2: Row Availability ---
Rows with active search signals: 30000 / 30000

--- Query 3: Target Distribution ---
target_priority_flag
False    18248
True     11752
Name: count, dtype: int64

--- Honest Model Precision: 0.6074 ---
--- TRAP Model Precision (With Leaked Feature): 0.9999 ---
Trap sprung! The score jumped to near-perfect because 'trend_pct' leaks the target label.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# Limitation: This snapshot uses a static 90-day window. Multi-year seasonality is not captured in this slice.
# Careful Words: We observe historical correlations between search position and traffic drop-off; we do not claim direct causality.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.